# Qualidade da camada Silver

Verificações usadas antes da criação da Gold.

In [0]:
from pyspark.sql import functions as F

partidas = spark.table("workspace.silver.partidas")
gols = spark.table("workspace.silver.gols")
cartoes = spark.table("workspace.silver.cartoes")
estatisticas = spark.table("workspace.silver.estatisticas")

## Contagens

In [0]:
contagens = [
    ("partidas", partidas.count()),
    ("gols", gols.count()),
    ("cartoes", cartoes.count()),
    ("estatisticas", estatisticas.count()),
]

display(spark.createDataFrame(contagens, ["tabela", "registros"]))

tabela,registros
partidas,9165
gols,10820
cartoes,20953
estatisticas,18330


## Chaves e relacionamentos

In [0]:
ids_partidas = partidas.select("partida_id")

duplicadas = partidas.groupBy("partida_id").count().filter("count > 1").count()
gols_orfaos = gols.join(ids_partidas, "partida_id", "left_anti").count()
cartoes_orfaos = cartoes.join(ids_partidas, "partida_id", "left_anti").count()
estatisticas_orfas = estatisticas.join(ids_partidas, "partida_id", "left_anti").count()
estatisticas_fora_do_grao = (
    estatisticas.groupBy("partida_id").count().filter("count <> 2").count()
)

integridade = [
    ("IDs duplicados em partidas", duplicadas),
    ("Gols sem partida", gols_orfaos),
    ("Cartões sem partida", cartoes_orfaos),
    ("Estatísticas sem partida", estatisticas_orfas),
    ("Partidas sem duas linhas de estatísticas", estatisticas_fora_do_grao),
]

display(spark.createDataFrame(integridade, ["teste", "ocorrencias"]))

teste,ocorrencias
IDs duplicados em partidas,0
Gols sem partida,0
Cartões sem partida,0
Estatísticas sem partida,0
Partidas sem duas linhas de estatísticas,0


## Campos principais

In [0]:
campos = [
    ("partida_id nulo", partidas.filter("partida_id IS NULL").count()),
    ("data nula", partidas.filter("data_partida IS NULL").count()),
    ("mandante vazio", partidas.filter("mandante IS NULL OR trim(mandante) = ''").count()),
    ("visitante vazio", partidas.filter("visitante IS NULL OR trim(visitante) = ''").count()),
    ("placar negativo", partidas.filter("placar_mandante < 0 OR placar_visitante < 0").count()),
    ("posse inválida", estatisticas.filter("posse_bola_pct < 0 OR posse_bola_pct > 100").count()),
    ("precisão inválida", estatisticas.filter("precisao_passes_pct < 0 OR precisao_passes_pct > 100").count()),
    ("minuto de gol inválido", gols.filter("minuto_base IS NULL").count()),
    ("minuto de cartão inválido", cartoes.filter("minuto_base IS NULL").count()),
]

display(spark.createDataFrame(campos, ["teste", "ocorrencias"]))

teste,ocorrencias
partida_id nulo,0
data nula,0
mandante vazio,0
visitante vazio,0
placar negativo,0
posse inválida,0
precisão inválida,0
minuto de gol inválido,0
minuto de cartão inválido,0


## Cobertura das estatísticas

In [0]:
cobertura = (
    estatisticas.join(partidas.select("partida_id", F.year("data_partida").alias("ano")), "partida_id")
    .groupBy("ano")
    .agg(
        F.count("*").alias("linhas"),
        F.count("posse_bola_pct").alias("com_posse"),
        F.count("precisao_passes_pct").alias("com_precisao_passes"),
        F.sum(F.col("estatisticas_disponiveis").cast("int")).alias("disponiveis")
    )
    .withColumn("posse_ausente", F.col("linhas") - F.col("com_posse"))
    .orderBy("ano")
)

display(cobertura)

ano,linhas,com_posse,com_precisao_passes,disponiveis,posse_ausente
2003,1104,0,0,0,1104
2004,1104,0,0,0,1104
2005,924,0,0,0,924
2006,760,0,0,0,760
2007,760,0,0,0,760
2008,760,0,0,0,760
2009,760,0,0,0,760
2010,760,0,0,0,760
2011,760,0,0,0,760
2012,760,0,0,0,760


A posse está ausente entre 2003 e 2013 e também em 2024. Há cobertura parcial em 2014, 2017, 2019, 2020 e 2025. Por isso, as análises de estatísticas usam somente partidas com a métrica disponível.

## Placar e eventos de gols

In [0]:
primeira_partida_com_eventos = gols.agg(F.min("partida_id")).first()[0]

gols_por_clube = gols.groupBy("partida_id", "clube").count()

placares = (
    partidas.filter(F.col("partida_id") >= primeira_partida_com_eventos).alias("p")
    .join(
        gols_por_clube.alias("gm"),
        (F.col("p.partida_id") == F.col("gm.partida_id")) & (F.col("p.mandante") == F.col("gm.clube")),
        "left"
    )
    .join(
        gols_por_clube.alias("gv"),
        (F.col("p.partida_id") == F.col("gv.partida_id")) & (F.col("p.visitante") == F.col("gv.clube")),
        "left"
    )
    .select(
        F.col("p.partida_id"),
        F.col("p.placar_mandante"),
        F.col("p.placar_visitante"),
        F.coalesce(F.col("gm.count"), F.lit(0)).alias("gols_mandante"),
        F.coalesce(F.col("gv.count"), F.lit(0)).alias("gols_visitante")
    )
)

placares_divergentes = placares.filter(
    (F.col("placar_mandante") != F.col("gols_mandante")) |
    (F.col("placar_visitante") != F.col("gols_visitante"))
)

resumo_placar = [
    ("Primeiro ID com eventos", primeira_partida_com_eventos),
    ("Partidas verificadas", placares.count()),
    ("Placares divergentes", placares_divergentes.count()),
]

display(spark.createDataFrame(resumo_placar, ["medida", "valor"]))

medida,valor
Primeiro ID com eventos,4607
Partidas verificadas,4559
Placares divergentes,0


## Outliers

O intervalo interquartil sinaliza valores extremos. Para as estatísticas, entram apenas linhas com dados disponíveis, evitando tratar ausências preenchidas com zero como observações reais.

In [0]:
gols_partida = (
    partidas.select(F.col("placar_mandante").cast("double").alias("valor"))
    .union(partidas.select(F.col("placar_visitante").cast("double").alias("valor")))
)

metricas_outliers = {
    "gols por clube": gols_partida,
    "chutes": estatisticas.filter("estatisticas_disponiveis").select(F.col("chutes").cast("double").alias("valor")),
    "chutes no alvo": estatisticas.filter("estatisticas_disponiveis").select(F.col("chutes_no_alvo").cast("double").alias("valor")),
    "passes": estatisticas.filter("estatisticas_disponiveis").select(F.col("passes").cast("double").alias("valor")),
    "faltas": estatisticas.filter("estatisticas_disponiveis").select(F.col("faltas").cast("double").alias("valor")),
    "cartões amarelos": estatisticas.filter("estatisticas_disponiveis").select(F.col("cartoes_amarelos").cast("double").alias("valor")),
    "cartões vermelhos": estatisticas.filter("estatisticas_disponiveis").select(F.col("cartoes_vermelhos").cast("double").alias("valor")),
    "impedimentos": estatisticas.filter("estatisticas_disponiveis").select(F.col("impedimentos").cast("double").alias("valor")),
    "escanteios": estatisticas.filter("estatisticas_disponiveis").select(F.col("escanteios").cast("double").alias("valor")),
}

resumo_outliers = []

for metrica, valores in metricas_outliers.items():
    valores = valores.filter("valor IS NOT NULL")
    q1, q3 = valores.approxQuantile("valor", [0.25, 0.75], 0.01)
    limite = q3 + 1.5 * (q3 - q1)
    quantidade = valores.filter(F.col("valor") > limite).count()
    maximo = valores.agg(F.max("valor")).first()[0]
    resumo_outliers.append((metrica, q1, q3, round(limite, 1), quantidade, maximo))

display(spark.createDataFrame(
    resumo_outliers,
    ["metrica", "q1", "q3", "limite_superior", "acima_limite", "maximo"]
))

metrica,q1,q3,limite_superior,acima_limite,maximo
gols por clube,0.0,2.0,5.0,58,8.0
chutes,9.0,16.0,26.5,78,37.0
chutes no alvo,1.0,5.0,11.0,36,18.0
passes,345.0,483.0,690.0,49,848.0
faltas,12.0,18.0,27.0,39,34.0
cartões amarelos,1.0,3.0,6.0,37,10.0
cartões vermelhos,0.0,0.0,0.0,882,3.0
impedimentos,1.0,2.0,3.5,833,10.0
escanteios,3.0,7.0,13.0,93,23.0


Os limites são alertas estatísticos, não regras do futebol. Os máximos encontrados permanecem em faixas possíveis para uma partida. Os registros foram mantidos para não apagar partidas reais ou eventos raros; não houve remoção nem substituição de valores extremos.

## Resultado

In [0]:
falhas_criticas = sum(valor for _, valor in integridade + campos) + placares_divergentes.count()

resultado = "aprovada" if falhas_criticas == 0 else "revisar"
display(spark.createDataFrame([(resultado, falhas_criticas)], ["situacao", "falhas_criticas"]))

situacao,falhas_criticas
aprovada,0
